# 06 · `--seed-offset` 这个参数不存在（HANDOFF §5 的指令有误）

**对应 HANDOFF §5**：

> 每个 run 建议 ≥3–5 seed(`--seed-offset`)。

**slime 的 `arguments.py` 里没有 `--seed-offset`。** 它有两个 seed 参数，
但都不叫这个名字：`--seed`（默认 1234）和 `--rollout-seed`。
按 HANDOFF 的写法传 `--seed-offset`，argparse 会直接报错退出。


In [1]:
%matplotlib inline
import json, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "font.size": 9,
    "axes.grid": True, "grid.alpha": .25,
    "axes.spines.top": False, "axes.spines.right": False,
})
R = Path("/lus/lfs1aip2/projects/public/u6gb/tasks/large-discovery-model/ldm_rl/results")
def load(name): return json.loads((R / name).read_text())

import subprocess
S = Path("/lus/lfs1aip2/projects/public/u6gb/tasks/large-discovery-model/LDM-rl/rl/slime")
hits = subprocess.run(
    ["grep", "-rn", "seed", str(S / "slime" / "utils" / "arguments.py")],
    capture_output=True, text=True).stdout
lines = [l for l in hits.splitlines() if "add_argument" in l or "seed" in l.lower()][:12]
print(f"slime/utils/arguments.py 里与 seed 有关的行：{len(lines)} 条\n")
for l in lines: print("  " + l.strip()[:110])

slime/utils/arguments.py 里与 seed 有关的行：3 条

  419:                "--rollout-seed",
  423:                    "The seed for the random number generator during rollout. "
  877:            reset_arg(parser, "--seed", type=int, default=1234)


## 真正的 seed 在哪

环境侧的 seed **烘焙在每条 episode 的 prompt JSON 里**
（`EpisodeSpec` 的 `"seed"` 字段），命令行覆盖不了它。要改只能重新生成
episodes 文件：

```bash
python -m ldm_rl.episodes --seed-offset <N>   # 这里的 --seed-offset 是有的
```

好消息是这一步**纯文本生成、不跑 docking**，16 行、24 KB，秒级完成。

编排脚本已经按这个方式处理：每个 run 拿到自己的 `episodes.jsonl`，
里面同时改掉了 `seed`、`gp_history_file` 和 `output_dir` 三样——
后两样跨 run 共享会让各 run 的 reward 变成彼此的函数，且结果取决于
并发时序、不可复现。

## 三个 seed 是三件不同的事

| 参数 | 在哪 | 管什么 |
|---|---|---|
| `--seed` | slime，默认 1234 | 框架侧的随机性（初始化、数据顺序） |
| `--rollout-seed` | slime | rollout 采样时的随机数 |
| `"seed"` 字段 | **episode 的 prompt JSON** | 环境侧：候选采样、GP、评测顺序 |

**HANDOFF 想表达的是第三个**，而它命令行改不了。
换 seed 换的是哪一个，决定了 R1–R4 的重复到底在重复什么——
只改 slime 的 seed 而环境 seed 不变，四个「不同种子」的 run
会走同一条环境轨迹。

## 建议 HANDOFF 改成

> 每个 run 用 `python -m ldm_rl.episodes --seed-offset <N>` **重新生成
> episodes 文件**来换环境 seed；slime 的 `--seed` / `--rollout-seed`
> 只管框架侧的随机性，改它们不会改变环境的采样。